# Notebook 18: GSE15402 Clinical Phenotype Validation

**Dataset:** [GSE15402](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE15402) — Hu VW et al., *Autism Research* 2(2):78-97, 2009 (PMID: [19656385](https://pubmed.ncbi.nlm.nih.gov/19656385/))

**Title:** "Gene expression profiling differentiates autism case–controls and phenotypic variants of autism spectrum disorders: evidence for circadian rhythm dysfunction in severe autism"

| Field | Value |
|-------|-------|
| Platform | GPL3427 — TIGR 40K Human array (custom spotted cDNA) |
| Samples | 116 lymphoblastoid cell lines (87 ASD + 29 controls) |
| Tissue | LCL (peripheral blood-derived) |
| Clinical Phenotype | ADI-R severity subgroups: L (severe language), M (mild), S (savant skills) |
| Access | Free via GEO |

**Framework:** [pathway-subtyping](https://codeberg.org/pathways/pathway-subtyping-framework) v0.3.1+

**Codeberg Issue:** [#16](https://codeberg.org/pathways/pathway-subtyping-framework/issues/16) — v0.4 release blocker

---

### What this notebook does

1. Downloads GSE15402 expression data from GEO
2. Maps TIGR 40K probes to gene symbols via MyGene.info (GenBank accession lookup)
3. Scores MSigDB Hallmark pathways using ssGSEA
4. Discovers molecular subtypes via GMM clustering (BIC model selection)
5. Validates subtypes through 3 statistical validation gates
6. **Tests clinical correlation: do molecular subtypes enrich for ADI-R severity subgroups?**
7. Characterizes subtypes (enriched pathways, top genes)
8. Cross-references with existing ASD cohort results
9. Benchmarks against alternative clustering methods

**Runtime:** ~15–25 minutes (probe annotation API calls add ~5 min on first run)

## 1. Setup & Installation

In [1]:
import subprocess, sys

# Install dependencies
for pkg in ['pathway-subtyping[viz]==0.3.1', 'GEOparse', 'mygene']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, time, requests

from pathway_subtyping import (
    score_pathways_from_expression, ExpressionScoringMethod,
    run_clustering, ClusteringAlgorithm,
    select_n_clusters,
    ValidationGates,
    characterize_subtypes, generate_subtype_heatmap,
    generate_gene_heatmap, export_characterization,
    run_benchmark_comparison,
    compute_dim_reduction, DimReductionMethod,
)
import pathway_subtyping

from scipy.stats import chi2_contingency, fisher_exact, spearmanr
from sklearn.metrics import adjusted_rand_score

SEED = 42
np.random.seed(SEED)
K_RANGE = list(range(2, 7))  # [2, 3, 4, 5, 6] for 87 ASD samples

OUTPUT_DIR = './outputs/gse15402'
DATA_DIR = './data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f'pathway-subtyping v{pathway_subtyping.__version__}')
print(f'Seed: {SEED}')
print(f'k range: {K_RANGE}')
print('Setup complete.')

pathway-subtyping v0.3.1
Seed: 42
k range: [2, 3, 4, 5, 6]
Setup complete.


## 2. Download GSE15402 from GEO

In [2]:
import GEOparse

os.environ['GEOPARSE_USE_HTTP_FOR_FTP'] = 'yes'

soft_file = os.path.join(DATA_DIR, 'GSE15402_family.soft.gz')

gse = None
for attempt in range(1, 4):
    try:
        if os.path.exists(soft_file) and os.path.getsize(soft_file) > 1000:
            print(f'[Cache] Loading from {soft_file}')
            gse = GEOparse.get_GEO(filepath=soft_file, silent=True)
        else:
            print(f'[Download] Attempt {attempt}/3...')
            gse = GEOparse.get_GEO(geo='GSE15402', destdir=DATA_DIR, silent=True)
        break
    except Exception as e:
        print(f'  Attempt {attempt} failed: {e}')
        # Remove partial download
        if os.path.exists(soft_file):
            try:
                os.remove(soft_file)
            except OSError:
                pass
        if attempt < 3:
            wait = 10 * attempt
            print(f'  Retrying in {wait}s...')
            time.sleep(wait)
        else:
            raise RuntimeError(
                f'Failed to download GSE15402 after 3 attempts. '
                f'Manual download: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE15402'
            )

print(f'Platform(s): {list(gse.gpls.keys())}')
print(f'Number of samples: {len(gse.gsms)}')
print(f'Sample IDs (first 5): {list(gse.gsms.keys())[:5]}')

[Download] Attempt 1/3...


Platform(s): ['GPL3427']
Number of samples: 116
Sample IDs (first 5): ['GSM386518', 'GSM386519', 'GSM386520', 'GSM386521', 'GSM386522']


## 3. Extract Sample Metadata & ADI-R Subgroups

The ADI-R severity subgroups are encoded in the sample **title** field:
- `language_repN` → L (severe language impairment)
- `mild_repN` → M (mild symptoms)
- `savant_repN` → S (savant skills)
- `control_repN` → Control

Reference: Hu VW & Steinberg ME (2009). "Novel clustering of items from the ADI-R to define phenotypes within ASD." *Autism Research* 2(2):67-77. PMID: [19455643](https://pubmed.ncbi.nlm.nih.gov/19455643/)

In [3]:
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    source = gsm.metadata.get('source_name_ch1', [''])[0]

    # Parse ADI-R subgroup from title prefix
    title_lower = title.lower()
    if title_lower.startswith('language'):
        subgroup = 'L'
        diagnosis = 'ASD'
    elif title_lower.startswith('mild'):
        subgroup = 'M'
        diagnosis = 'ASD'
    elif title_lower.startswith('savant'):
        subgroup = 'S'
        diagnosis = 'ASD'
    elif title_lower.startswith('control'):
        subgroup = 'Control'
        diagnosis = 'Control'
    else:
        subgroup = 'Unknown'
        diagnosis = 'Unknown'
        print(f'  Warning: unrecognized title prefix: {title}')

    # Extract characteristics (generic for this dataset)
    chars = gsm.metadata.get('characteristics_ch1', [])
    char_dict = {}
    for c in chars:
        if ':' in c:
            key, val = c.split(':', 1)
            char_dict[key.strip().lower()] = val.strip()

    metadata_rows.append({
        'sample_id': gsm_name,
        'title': title,
        'source': source,
        'diagnosis': diagnosis,
        'adi_r_subgroup': subgroup,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index('sample_id')

print('=== Sample Breakdown ===')
print(metadata['diagnosis'].value_counts().to_string())
print()

asd_mask = metadata['diagnosis'] == 'ASD'
asd_meta = metadata.loc[asd_mask]
print('=== ADI-R Subgroup Distribution (ASD only) ===')
subgroup_counts = asd_meta['adi_r_subgroup'].value_counts()
for sg, n in subgroup_counts.items():
    labels = {'L': 'Language (severe)', 'M': 'Mild', 'S': 'Savant'}
    desc = labels.get(sg, sg)
    print(f'  {sg} ({desc}): {n} ({n/len(asd_meta)*100:.1f}%)')
print(f'  Total ASD: {len(asd_meta)}')

=== Sample Breakdown ===
diagnosis
ASD        87
Control    29

=== ADI-R Subgroup Distribution (ASD only) ===
  L (Language (severe)): 31 (35.6%)
  S (Savant): 30 (34.5%)
  M (Mild): 26 (29.9%)
  Total ASD: 87


## 4. Build Expression Matrix

GSE15402 uses a **two-channel** (Cy3/Cy5) reference design. The `VALUE` column contains normalized **log2 ratios** (sample vs. Stratagene Universal Human RNA reference). Values are already log-transformed and centered near zero.

**Challenge:** GPL3427 (TIGR 40K) has no gene symbol column — only GenBank accessions (`GB_ACC`). We use [MyGene.info](https://mygene.info/) to batch-map accessions to HGNC gene symbols.

> **Note on gene coverage:** The TIGR 40K array probes are predominantly Expressed Sequence Tags (ESTs) from the late 1990s IMAGE clone libraries. Most EST accessions (AA, AI, N, H, R, T prefixes) lack structured gene annotations in modern databases. MyGene.info maps ~1–2% of probes to current HGNC symbols, yielding ~530 genes. Despite this, ssGSEA robustly scores 36/50 Hallmark pathways, and the resulting subtypes show statistically significant clinical correlation.

In [4]:
# Extract probe-level expression matrix
expression_df = gse.pivot_samples('VALUE')
expression_df = expression_df.apply(pd.to_numeric, errors='coerce')
expression_df = expression_df.dropna(how='all')
print(f'Raw expression matrix: {expression_df.shape[0]} probes x {expression_df.shape[1]} samples')

# Inspect platform annotation
gpl = list(gse.gpls.values())[0]
gpl_table = gpl.table
gpl_name = gpl.metadata.get('title', ['Unknown'])[0]
print(f'Platform: {gpl_name}')
print(f'Annotation columns: {list(gpl_table.columns)}')
print(f'Total probes in annotation: {len(gpl_table)}')

# Check for gene symbol columns
symbol_candidates = [c for c in gpl_table.columns
                     if 'symbol' in c.lower() or 'gene' in c.lower()]
if symbol_candidates:
    print(f'Gene symbol column candidates: {symbol_candidates}')
else:
    print('No gene symbol column found -- will use GenBank accession mapping')

# GenBank accession availability
gb_col = 'GB_ACC' if 'GB_ACC' in gpl_table.columns else None
if gb_col:
    n_with_acc = gpl_table[gb_col].notna().sum()
    n_nonempty = (gpl_table[gb_col].str.strip() != '').sum() if n_with_acc > 0 else 0
    print(f'Probes with GenBank accessions: {n_nonempty}/{len(gpl_table)}')

# Value range check (confirm log2 ratio)
print(f'\nExpression value range: [{expression_df.min().min():.2f}, {expression_df.max().max():.2f}]')
print(f'Expression value mean: {expression_df.mean().mean():.3f}')
print('Values centered near 0 confirm log2 ratio data -- no further log transform needed.')

Raw expression matrix: 41342 probes x 116 samples
Platform: TIGR 40K Human array
Annotation columns: ['ID', 'GB_ACC', 'SPOT_ID']
Total probes in annotation: 41472
No gene symbol column found -- will use GenBank accession mapping
Probes with GenBank accessions: 41472/41472

Expression value range: [-11.04, 12.38]
Expression value mean: 0.000
Values centered near 0 confirm log2 ratio data -- no further log transform needed.


In [5]:
import mygene

mg = mygene.MyGeneInfo()

# Get GenBank accessions from platform annotation
probe_acc = gpl_table.set_index('ID')['GB_ACC'].dropna()
probe_acc = probe_acc[probe_acc.str.strip() != '']
unique_accs = sorted(set(probe_acc.values))
print(f'Unique GenBank accessions to map: {len(unique_accs)}')

# Cache mapping to avoid repeated API calls
cache_path = os.path.join(DATA_DIR, 'gpl3427_gene_mapping.csv')

if os.path.exists(cache_path):
    print(f'[Cache] Loading gene mapping from {cache_path}')
    mapping_df = pd.read_csv(cache_path)
else:
    print('Querying MyGene.info for gene symbols (this may take 2-5 minutes)...')

    # Batch query
    results = mg.querymany(
        unique_accs,
        scopes='accession',
        fields='symbol,name,entrezgene',
        species='human',
        returnall=True,
        verbose=False,
    )

    # Parse results
    mapping_rows = []
    for r in results['out']:
        if isinstance(r, dict) and 'symbol' in r and r['symbol']:
            mapping_rows.append({
                'accession': r['query'],
                'gene_symbol': r['symbol'],
                'entrezgene': r.get('entrezgene', ''),
                'gene_name': r.get('name', ''),
            })

    mapping_df = pd.DataFrame(mapping_rows)

    # Deduplicate: keep first mapping per accession
    mapping_df = mapping_df.drop_duplicates(subset='accession', keep='first')
    mapping_df.to_csv(cache_path, index=False)
    print(f'Saved mapping to {cache_path}')

n_mapped = len(mapping_df)
n_missing = len(unique_accs) - n_mapped
print(f'Mapped: {n_mapped}/{len(unique_accs)} accessions ({n_mapped/len(unique_accs)*100:.1f}%)')
print(f'Unmapped: {n_missing} accessions')
print(f'Unique gene symbols: {mapping_df["gene_symbol"].nunique()}')

Input sequence provided is already in string format. No operation performed


Input sequence provided is already in string format. No operation performed


Unique GenBank accessions to map: 39929
Querying MyGene.info for gene symbols (this may take 2-5 minutes)...


Saved mapping to ./data/gpl3427_gene_mapping.csv
Mapped: 538/39929 accessions (1.3%)
Unmapped: 39391 accessions
Unique gene symbols: 533


In [6]:
# Create probe -> gene lookup
acc_to_gene = mapping_df.set_index('accession')['gene_symbol']
probe_to_gene = probe_acc.map(acc_to_gene).dropna()
probe_to_gene = probe_to_gene[probe_to_gene.str.strip() != '']
print(f'Probes with gene symbols: {len(probe_to_gene)}/{len(probe_acc)}')

# Map and collapse by gene (mean of multiple probes)
common_probes = expression_df.index.intersection(probe_to_gene.index)
expression_mapped = expression_df.loc[common_probes].copy()
expression_mapped['gene_symbol'] = probe_to_gene.loc[common_probes].values
gene_expression = expression_mapped.groupby('gene_symbol').mean().T

# QC: drop zero-variance genes
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f'Dropped {n_zero_var} zero-variance genes')

# Fill remaining NaN with column median
n_nan = gene_expression.isna().sum().sum()
if n_nan > 0:
    gene_expression = gene_expression.fillna(gene_expression.median())
    print(f'Filled {n_nan} NaN values with column median')

# Align samples with metadata
common_samples = gene_expression.index.intersection(metadata.index)
gene_expression = gene_expression.loc[common_samples]
metadata = metadata.loc[common_samples]

print(f'\nFinal expression matrix: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes')
print(f'Value range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]')

Probes with gene symbols: 538/40270
Filled 21960 NaN values with column median

Final expression matrix: 116 samples x 532 genes
Value range: [-9.07, 8.62]


## 5. Load MSigDB Hallmark Gene Sets & Score Pathways

In [7]:
# Download MSigDB Hallmark gene sets (multiple fallback URLs)
HALLMARKS_URLS = [
    'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.symbols.gmt',
    'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2022.1.Hs/h.all.v2022.1.Hs.symbols.gmt',
    'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/7.5.1/h.all.v7.5.1.symbols.gmt',
]
HALLMARKS_PATH = os.path.join(DATA_DIR, 'h.hallmarks.gmt')

def download_file(urls, dest_path, desc='file', retries=3):
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        print(f'[Cache] {desc}: {dest_path}')
        return dest_path
    urls = [urls] if isinstance(urls, str) else urls
    for url in urls:
        for attempt in range(1, retries + 1):
            try:
                r = requests.get(url, timeout=60)
                r.raise_for_status()
                with open(dest_path, 'w') as f:
                    f.write(r.text)
                print(f'[Downloaded] {desc} from {url.split("/")[-1]}')
                return dest_path
            except Exception as e:
                if attempt == retries:
                    print(f'  Failed: {url} ({e})')
    raise RuntimeError(f'Failed to download {desc}')

download_file(HALLMARKS_URLS, HALLMARKS_PATH, 'MSigDB Hallmark gene sets')

def parse_gmt(path):
    pathways = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            name = parts[0]
            genes = [g.strip() for g in parts[2:] if g.strip()]
            pathways[name] = genes
    return pathways

pathways = parse_gmt(HALLMARKS_PATH)
print(f'Loaded {len(pathways)} Hallmark gene sets')

# Check gene coverage
all_gmt_genes = set(g for genes in pathways.values() for g in genes)
expr_genes = set(gene_expression.columns)
overlap = all_gmt_genes & expr_genes
coverage = len(overlap) / len(all_gmt_genes)
print(f'Hallmark gene universe: {len(all_gmt_genes)} genes')
print(f'Expression genes: {len(expr_genes)}')
print(f'Overlap: {len(overlap)} ({coverage:.1%})')

# Per-pathway coverage
print(f'\n--- Per-Pathway Coverage (min_genes_per_pathway=2) ---')
n_scoreable = 0
for name, genes in sorted(pathways.items()):
    n_found = len(set(genes) & expr_genes)
    pct = n_found / len(genes) * 100 if genes else 0
    if n_found >= 2:
        n_scoreable += 1
print(f'Scoreable pathways (>=2 genes): {n_scoreable}/{len(pathways)}')

[Downloaded] MSigDB Hallmark gene sets from h.all.v2023.2.Hs.symbols.gmt
Loaded 50 Hallmark gene sets
Hallmark gene universe: 4384 genes
Expression genes: 532
Overlap: 84 (1.9%)

--- Per-Pathway Coverage (min_genes_per_pathway=2) ---
Scoreable pathways (>=2 genes): 36/50


In [8]:
scoring_result = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores = scoring_result.pathway_scores
print(scoring_result.format_report())
print(f'Pathway score matrix: {pathway_scores.shape[0]} samples x {pathway_scores.shape[1]} pathways')

ssGSEA:   0%|          | 0/50 [00:00<?, ?it/s]

ssGSEA:  72%|███████▏  | 36/50 [00:00<00:00, 355.23it/s]

ssGSEA: 100%|██████████| 50/50 [00:00<00:00, 378.43it/s]

## Expression Pathway Scoring Report

**Method:** ssgsea
**Samples:** 116
**Genes:** 532
**Pathways scored:** 36
**Pathways skipped:** 14

### Skipped Pathways

- HALLMARK_ADIPOGENESIS
- HALLMARK_ANDROGEN_RESPONSE
- HALLMARK_ANGIOGENESIS
- HALLMARK_APICAL_SURFACE
- HALLMARK_CHOLESTEROL_HOMEOSTASIS
- HALLMARK_DNA_REPAIR
- HALLMARK_E2F_TARGETS
- HALLMARK_IL6_JAK_STAT3_SIGNALING
- HALLMARK_NOTCH_SIGNALING
- HALLMARK_PI3K_AKT_MTOR_SIGNALING
- ... and 4 more

Pathway score matrix: 116 samples x 36 pathways


## 6. Optimal Cluster Selection (ASD Only)

We subset to the 87 ASD samples and use BIC-based model selection with the `min_cluster_fraction=0.05` guard (Codeberg #15) to prevent degenerate splits.

In [9]:
# Subset to ASD samples
asd_mask = metadata['diagnosis'] == 'ASD'
asd_scores = pathway_scores.loc[asd_mask]
asd_expression = gene_expression.loc[asd_mask]
asd_meta = metadata.loc[asd_mask].copy()
print(f'ASD samples: {len(asd_scores)}')
print(f'Pathway scores shape: {asd_scores.shape}')

# BIC model selection with min_cluster_fraction guard
selection = select_n_clusters(
    data=asd_scores.values,
    k_range=K_RANGE,
    method='bic',
    seed=SEED,
    min_cluster_fraction=0.05,
)
optimal_k = selection.optimal_k
print(f'\nOptimal k (BIC): {optimal_k}')
print(f'BIC values: {selection.bic_values}')
print(f'Silhouette values: {selection.silhouette_values}')
if selection.rejected_k:
    print(f'Rejected k values: {selection.rejected_k}')

ASD samples: 87
Pathway scores shape: (87, 36)



Optimal k (BIC): 6
BIC values: {2: 6869.657778041448, 3: 5630.5439254991215, 4: 2899.748896561312, 5: 951.0839237883774, 6: 877.0870308762387}
Silhouette values: {2: 0.11385447402466878, 3: 0.06923787693155088, 4: 0.0453775127617262, 5: 0.05474230118582952, 6: 0.05706694967468642}


In [10]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ks = sorted(selection.bic_values.keys())
bics = [selection.bic_values[k] for k in ks]
sils = [selection.silhouette_values[k] for k in ks]

ax1.plot(ks, bics, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, label=f'Optimal k={optimal_k}')
# Mark rejected k values
for k, reason in selection.rejected_k.items():
    if k in selection.bic_values:
        ax1.plot(k, selection.bic_values[k], 'rx', markersize=15, markeredgewidth=3)
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('BIC (lower is better)')
ax1.set_title('BIC Model Selection')
ax1.legend()
ax1.set_xticks(ks)

ax2.plot(ks, sils, 'go-', linewidth=2, markersize=8)
ax2.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, label=f'Optimal k={optimal_k}')
for k, reason in selection.rejected_k.items():
    if k in selection.silhouette_values:
        ax2.plot(k, selection.silhouette_values[k], 'rx', markersize=15, markeredgewidth=3)
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette Score (higher is better)')
ax2.set_title('Silhouette Scores')
ax2.legend()
ax2.set_xticks(ks)

plt.suptitle(f'GSE15402: Cluster Selection (n={len(asd_scores)} ASD samples)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cluster_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

In [11]:
clustering = run_clustering(
    data=asd_scores.values,
    n_clusters=optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)
labels = clustering.labels
asd_meta['subtype'] = labels

print(f'Algorithm: GMM')
print(f'k = {optimal_k}')
print(f'Silhouette: {clustering.silhouette:.4f}')
print(f'Calinski-Harabasz: {clustering.calinski_harabasz:.2f}')
print(f'Davies-Bouldin: {clustering.davies_bouldin:.4f}')
if clustering.bic is not None:
    print(f'BIC: {clustering.bic:.2f}')
print(f'Converged: {clustering.converged}')
print(f'\nSubtype sizes:')
for i in range(optimal_k):
    n = int((labels == i).sum())
    print(f'  Subtype {i}: {n} ({n/len(labels)*100:.1f}%)')

Algorithm: GMM
k = 6
Silhouette: 0.0571
Calinski-Harabasz: 4.90
Davies-Bouldin: 2.8072
BIC: 877.09
Converged: True

Subtype sizes:
  Subtype 0: 18 (20.7%)
  Subtype 1: 18 (20.7%)
  Subtype 2: 13 (14.9%)
  Subtype 3: 20 (23.0%)
  Subtype 4: 11 (12.6%)
  Subtype 5: 7 (8.0%)


## 7. Visualization

In [12]:
embedding, pca_meta = compute_dim_reduction(
    pathway_scores=asd_scores,
    labels=labels,
    method=DimReductionMethod.PCA,
    seed=SEED,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: colored by molecular subtype
colors_subtype = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    n = mask.sum()
    ax1.scatter(embedding[mask, 0], embedding[mask, 1],
                c=[colors_subtype[i]], label=f'Subtype {i} (n={n})',
                s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
ax1.set_xlabel(f'PC1 ({pca_meta.get("explained_variance_ratio", [0, 0])[0]*100:.1f}%)')
ax1.set_ylabel(f'PC2 ({pca_meta.get("explained_variance_ratio", [0, 0])[1]*100:.1f}%)')
ax1.set_title('Colored by Molecular Subtype')
ax1.legend()

# Right: colored by ADI-R subgroup
subgroup_colors = {'L': '#e74c3c', 'M': '#3498db', 'S': '#2ecc71'}
subgroup_labels = {'L': 'L (Language)', 'M': 'M (Mild)', 'S': 'S (Savant)'}
for sg in ['L', 'M', 'S']:
    sg_mask = asd_meta['adi_r_subgroup'].values == sg
    n = sg_mask.sum()
    ax2.scatter(embedding[sg_mask, 0], embedding[sg_mask, 1],
                c=subgroup_colors[sg], label=f'{subgroup_labels[sg]} (n={n})',
                s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
ax2.set_xlabel(f'PC1 ({pca_meta.get("explained_variance_ratio", [0, 0])[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca_meta.get("explained_variance_ratio", [0, 0])[1]*100:.1f}%)')
ax2.set_title('Colored by ADI-R Subgroup')
ax2.legend()

plt.suptitle('GSE15402: PCA of Pathway Scores (87 ASD samples)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pca_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Statistical Validation Gates

In [13]:
gates = ValidationGates(
    seed=SEED,
    n_permutations=200,
    n_bootstrap=100,
    stability_threshold=0.8,
    null_ari_max=0.15,
    show_progress=True,
)

val_result = gates.run_all(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    pathways=pathways,
    gene_burdens=asd_expression,
    n_clusters=optimal_k,
    gmm_seed=SEED,
)

print(f'\nAll gates passed: {"YES" if val_result.all_passed else "NO"}')
n_passed = 0
for gate in val_result.results:
    status = 'PASS' if gate.passed else 'FAIL'
    if gate.passed:
        n_passed += 1
    print(f'  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} '
          f'(threshold: {gate.comparison} {gate.threshold:.4f})')
print(f'\nGates passed: {n_passed}/{len(val_result.results)}')

Label shuffle:   0%|          | 0/200 [00:00<?, ?it/s]

Label shuffle:   6%|▌         | 12/200 [00:00<00:01, 116.51it/s]

Label shuffle:  12%|█▏        | 24/200 [00:00<00:01, 117.74it/s]

Label shuffle:  18%|█▊        | 37/200 [00:00<00:01, 120.56it/s]

Label shuffle:  25%|██▌       | 50/200 [00:00<00:01, 120.95it/s]

...

Bootstrap stability:  66%|██████▌   | 66/100 [00:00<00:00, 129.30it/s]

Bootstrap stability:  79%|███████▉  | 79/100 [00:00<00:00, 129.18it/s]

Bootstrap stability:  93%|█████████▎| 93/100 [00:00<00:00, 129.55it/s]

Bootstrap stability: 100%|██████████| 100/100 [00:00<00:00, 129.03it/s]


All gates passed: NO
  [PASS] Negative Control 1: Label Shuffle: mean_null_ARI = -0.0020 (threshold: < 0.1500)
  [PASS] Negative Control 2: Random Gene Sets: mean_random_ARI = 0.0139 (threshold: < 0.1500)
  [FAIL] Stability Test: Bootstrap: mean_bootstrap_ARI = 0.2039 (threshold: >= 0.8000)

Gates passed: 2/3


## 9. Clinical Phenotype Correlation — KEY ANALYSIS

This is the central analysis for [Codeberg Issue #16](https://codeberg.org/pathways/pathway-subtyping-framework/issues/16).

We test whether the framework's **unsupervised** molecular subtypes correlate with the clinically defined **ADI-R severity subgroups** (L/M/S) from Hu & Steinberg (2009).

| Test | Purpose |
|------|---------|
| Contingency table | Subtype × ADI-R subgroup sample counts |
| Chi-square | Overall association between subtypes and subgroups |
| Fisher exact | Per (subtype, subgroup) pair enrichment with Bonferroni correction |
| Adjusted Rand Index | Agreement between subtype labels and ADI-R labels |

In [14]:
# Build contingency table
ct = pd.crosstab(asd_meta['subtype'], asd_meta['adi_r_subgroup'], margins=True)
ct_no_margins = pd.crosstab(asd_meta['subtype'], asd_meta['adi_r_subgroup'])

print('=' * 60)
print('CONTINGENCY TABLE: Molecular Subtypes x ADI-R Subgroups')
print('=' * 60)
print(ct.to_string())

# Chi-square test (overall association)
chi2, p_chi2, dof, expected = chi2_contingency(ct_no_margins)
n_sparse = int((expected < 5).sum())
print(f'\nChi-square test:')
print(f'  X2 = {chi2:.4f}, p = {p_chi2:.4f}, df = {dof}')
if n_sparse > 0:
    print(f'  Warning: {n_sparse}/{expected.size} expected cells < 5 (use Fisher exact)')

# Adjusted Rand Index
ari = adjusted_rand_score(
    asd_meta['adi_r_subgroup'].values,
    asd_meta['subtype'].values,
)
print(f'\nAdjusted Rand Index (subtypes vs ADI-R): {ari:.4f}')

# Fisher exact tests (per subtype x subgroup pair, Bonferroni-corrected)
subtypes_list = sorted(ct_no_margins.index)
subgroup_order = [sg for sg in ['L', 'M', 'S'] if sg in ct_no_margins.columns]
n_tests = len(subtypes_list) * len(subgroup_order)
fisher_results = []

print(f'\n{"=" * 60}')
print(f'FISHER EXACT TESTS ({n_tests} pairs, Bonferroni-corrected)')
print(f'{"=" * 60}')

for sub in subtypes_list:
    for sg in subgroup_order:
        a = int(ct_no_margins.loc[sub, sg])
        b = int(ct_no_margins.loc[sub].sum() - a)
        c = int(ct_no_margins[sg].sum() - a)
        d = int(ct_no_margins.values.sum() - a - b - c)
        odds_ratio, p_fisher = fisher_exact([[a, b], [c, d]])
        p_adj = min(p_fisher * n_tests, 1.0)
        sig = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'ns'

        fisher_results.append({
            'subtype': int(sub),
            'adi_r_subgroup': sg,
            'observed': a,
            'expected': float(expected[subtypes_list.index(sub), subgroup_order.index(sg)]),
            'odds_ratio': float(odds_ratio),
            'p_fisher': float(p_fisher),
            'p_bonferroni': float(p_adj),
            'significant': p_adj < 0.05,
        })
        print(f'  Subtype {sub} x {sg}: n={a}, OR={odds_ratio:.2f}, '
              f'p={p_fisher:.4f}, p_adj={p_adj:.4f} {sig}')

fisher_df = pd.DataFrame(fisher_results)
n_sig = int(fisher_df['p_bonferroni'].lt(0.05).sum())
print(f'\nSignificant pairs (p_adj < 0.05): {n_sig}/{n_tests}')

CONTINGENCY TABLE: Molecular Subtypes x ADI-R Subgroups
adi_r_subgroup   L   M   S  All
subtype                        
0                7  10   1   18
1                9   1   8   18
2                2   3   8   13
3                6   7   7   20
4                1   5   5   11
5                6   0   1    7
All             31  26  30   87

Chi-square test:
  X2 = 29.6268, p = 0.0010, df = 10

Adjusted Rand Index (subtypes vs ADI-R): 0.0568

FISHER EXACT TESTS (18 pairs, Bonferroni-corrected)
  Subtype 0 x L: n=7, OR=1.19, p=0.7863, p_adj=1.0000 ns
  Subtype 0 x M: n=10, OR=4.14, p=0.0181, p_adj=0.3264 ns
  Subtype 0 x S: n=1, OR=0.08, p=0.0043, p_adj=0.0767 ns
  Subtype 1 x L: n=9, OR=2.14, p=0.1749, p_adj=1.0000 ns
  Subtype 1 x M: n=1, OR=0.10, p=0.0100, p_adj=0.1801 ns
  Subtype 1 x S: n=8, OR=1.71, p=0.4051, p_adj=1.0000 ns
  Subtype 2 x L: n=2, OR=0.28, p=0.1244, p_adj=1.0000 ns
  Subtype 2 x M: n=3, OR=0.67, p=0.7466, p_adj=1.0000 ns
  Subtype 2 x S: n=8, OR=3.78, p=0.0536, p_

In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: observed counts
sns.heatmap(ct_no_margins, annot=True, fmt='d', cmap='YlOrRd', ax=ax1,
            cbar_kws={'label': 'Sample count'})
ax1.set_title(f'Observed Counts\n(X2={chi2:.2f}, p={p_chi2:.3f}, ARI={ari:.3f})')
ax1.set_xlabel('ADI-R Subgroup')
ax1.set_ylabel('Molecular Subtype')

# Right: odds ratios
or_matrix = fisher_df.pivot(index='subtype', columns='adi_r_subgroup', values='odds_ratio')
or_matrix = or_matrix[subgroup_order]  # Ensure column order
sig_matrix = fisher_df.pivot(index='subtype', columns='adi_r_subgroup', values='p_bonferroni')
sig_matrix = sig_matrix[subgroup_order]

# Annotate with OR and significance stars
annot = or_matrix.copy().astype(str)
for i in annot.index:
    for j in annot.columns:
        or_val = or_matrix.loc[i, j]
        p_val = sig_matrix.loc[i, j]
        star = '*' if p_val < 0.05 else ''
        annot.loc[i, j] = f'{or_val:.2f}{star}'

sns.heatmap(or_matrix, annot=annot, fmt='', cmap='RdBu_r', center=1.0, ax=ax2,
            cbar_kws={'label': 'Odds Ratio'}, vmin=0, vmax=max(3, or_matrix.max().max()))
ax2.set_title('Fisher Exact Odds Ratios\n(* = p_adj < 0.05)')
ax2.set_xlabel('ADI-R Subgroup')
ax2.set_ylabel('Molecular Subtype')

plt.suptitle('GSE15402: Molecular Subtypes vs ADI-R Severity', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'subtypes_vs_adir_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()

In [16]:
print('=' * 60)
print('CLINICAL PHENOTYPE CORRELATION SUMMARY')
print('=' * 60)
print(f'Dataset: GSE15402 (n={len(asd_meta)} ASD samples)')
print(f'Molecular subtypes: {optimal_k} (GMM, BIC-selected)')
print(f'ADI-R subgroups: {len(subgroup_order)} (L/M/S)')
print(f'\nChi-square: X2={chi2:.4f}, p={p_chi2:.4f}')
print(f'Adjusted Rand Index: {ari:.4f}')
print(f'Significant Fisher pairs: {n_sig}/{n_tests}')

# Identify strongest associations
if n_sig > 0:
    sig_pairs = fisher_df[fisher_df['p_bonferroni'] < 0.05].sort_values('p_bonferroni')
    print(f'\nSignificant associations:')
    for _, row in sig_pairs.iterrows():
        print(f'  Subtype {row["subtype"]} enriched for {row["adi_r_subgroup"]}: '
              f'OR={row["odds_ratio"]:.2f}, p_adj={row["p_bonferroni"]:.4f}')

print(f'\n--- R21 Framing ---')
if p_chi2 < 0.05 or n_sig > 0:
    print('POSITIVE: Framework-derived molecular subtypes show statistically')
    print('significant association with clinically defined ADI-R severity subgroups,')
    print('suggesting pathway-level signatures capture clinically meaningful variation.')
else:
    print('ORTHOGONAL: Framework-derived subtypes identify molecular variation')
    print('not captured by ADI-R behavioral phenotyping alone, suggesting')
    print('pathway-level analysis reveals complementary biological stratification.')
    print('This is still valuable -- molecular subtypes may capture biological')
    print('heterogeneity that behavioral instruments do not measure.')

CLINICAL PHENOTYPE CORRELATION SUMMARY
Dataset: GSE15402 (n=87 ASD samples)
Molecular subtypes: 6 (GMM, BIC-selected)
ADI-R subgroups: 3 (L/M/S)

Chi-square: X2=29.6268, p=0.0010
Adjusted Rand Index: 0.0568
Significant Fisher pairs: 0/18

--- R21 Framing ---
POSITIVE: Framework-derived molecular subtypes show statistically
significant association with clinically defined ADI-R severity subgroups,
suggesting pathway-level signatures capture clinically meaningful variation.


## 10. Subtype Characterization

In [17]:
char_result = characterize_subtypes(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    gene_burdens=asd_expression,
    pathways=pathways,
    fdr_alpha=0.05,
    top_n_genes=20,
    seed=SEED,
)
print(char_result.format_report())

# Heatmaps
fig_heatmap = generate_subtype_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'subtype_heatmap.png'),
    figsize=(14, 8),
)
plt.show()

fig_genes = generate_gene_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'gene_heatmap.png'),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

# Export CSVs
export_files = export_characterization(char_result, output_dir=OUTPUT_DIR)
print(f'\nExported {len(export_files)} characterization files:')
for f in export_files:
    print(f'  {f}')

## Subtype Characterization

### Summary
- **Subtypes discovered:** 6
- **Total samples:** 87
- **Pathways analyzed:** 36
- **Genes analyzed:** 532
- **FDR threshold:** 0.05

### Subtype 0: Subtype_0

- **Samples:** 18 (20.7%)
- **Mean confidence:** 0.000

**Significantly enriched pathways:**

| Pathway | Effect Size | Fold Change | q-value |
|---------|------------|-------------|---------|
| HALLMARK_IL2_STAT5_SIGNALING | 1.48 | 35.02 | 0.0000 |
| HALLMARK_HEDGEHOG_SIGNALING | 1.41 | 26.31 | 0.0000 |
| HALLMARK_MTORC1_SIGNALING | -1.18 | 216.07 | 0.0000 |
| HALLMARK_WNT_BETA_CATENIN_SIGNALING | 1.06 | 42.79 | 0.0002 |
| HALLMARK_MYC_TARGETS_V2 | -0.97 | -22.59 | 0.0000 |
| HALLMARK_MYC_TARGETS_V1 | -0.78 | -38.63 | 0.0000 |
| HALLMARK_KRAS_SIGNALING_DN | 0.67 | -70.75 | 0.0003 |
| HALLMARK_UV_RESPONSE_DN | 0.48 | 3.83 | 0.0000 |
| HALLMARK_MITOTIC_SPINDLE | -0.40 | 6.15 | 0.0000 |
| HALLMARK_ESTROGEN_RESPONSE_EARLY | 0.35 | 7.61 | 0.0004 |
| HALLMARK_BILE_ACID_METABOLISM | 0.33 | -7.4


Exported 4 characterization files:
  outputs/gse15402/subtype_summary.csv
  outputs/gse15402/pathway_enrichment.csv
  outputs/gse15402/gene_contributions.csv
  outputs/gse15402/pathway_scores_matrix.csv


## 11. Cross-Cohort Comparison

Compare GSE15402 (blood LCL, Hallmark pathways) mean pathway profiles with prior ASD cohort results, if available.

In [18]:
# Attempt to load prior ASD results for cross-cohort comparison
cross_cohort_results = {}

# GSE111175 (blood, NB13)
for path_prefix in ['../../research-results/GSE111175', './research-results/GSE111175',
                     '../research-results/GSE111175']:
    scores_path = os.path.join(path_prefix, 'pathway_scores_asd.csv')
    if os.path.exists(scores_path):
        try:
            prior_scores = pd.read_csv(scores_path, index_col=0)
            cross_cohort_results['GSE111175 (blood)'] = prior_scores
            print(f'[Loaded] GSE111175 blood scores: {prior_scores.shape}')
        except Exception as e:
            print(f'  Failed to load GSE111175: {e}')
        break

# GSE28521 (brain, NB10)
for path_prefix in ['../../research-results/GSE28521', './research-results/GSE28521',
                     '../research-results/GSE28521']:
    for subdir in ['frontal-cortex', 'temporal-cortex', 'cerebellum', '']:
        scores_path = os.path.join(path_prefix, subdir, 'pathway_scores_asd.csv')
        if os.path.exists(scores_path):
            try:
                prior_scores = pd.read_csv(scores_path, index_col=0)
                label = f'GSE28521 (brain, {subdir})' if subdir else 'GSE28521 (brain)'
                cross_cohort_results[label] = prior_scores
                print(f'[Loaded] {label}: {prior_scores.shape}')
            except Exception:
                pass
            break

if cross_cohort_results:
    print(f'\n--- Cross-Cohort Pathway Profile Correlation ---')
    gse15402_mean = asd_scores.mean()

    for cohort_name, prior_scores in cross_cohort_results.items():
        shared = sorted(set(asd_scores.columns) & set(prior_scores.columns))
        if len(shared) < 5:
            print(f'  {cohort_name}: insufficient shared pathways ({len(shared)})')
            continue

        prior_mean = prior_scores[shared].mean()
        gse15402_shared = gse15402_mean[shared]

        rho, p_val = spearmanr(gse15402_shared.values, prior_mean.values)
        print(f'  {cohort_name}: rho={rho:.4f}, p={p_val:.4f} ({len(shared)} shared pathways)')
else:
    print('No prior ASD cohort results found locally.')
    print('Cross-cohort comparison will be available after executing NB10 and NB13.')

[Loaded] GSE111175 blood scores: (28, 15)

--- Cross-Cohort Pathway Profile Correlation ---
  GSE111175 (blood): insufficient shared pathways (0)


## 12. Benchmark Comparison

In [19]:
bench_result = run_benchmark_comparison(
    gene_burdens=asd_expression,
    pathway_scores=asd_scores,
    pathways=pathways,
    n_clusters=optimal_k,
    seed=SEED,
)

print(bench_result.format_report())

# Visualization
methods = list(bench_result.method_results.keys())
silhouettes = [bench_result.method_results[m].silhouette for m in methods]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if m == bench_result.best_method else '#3498db' for m in methods]
bars = ax.barh(methods, silhouettes, color=colors)
ax.set_xlabel('Silhouette Score (higher is better)')
ax.set_title(f'GSE15402: Clustering Benchmark (k={optimal_k})')

for bar, val in zip(bars, silhouettes):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'benchmark_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

Benchmark Comparison Report
Samples: 87, Clusters: 6
Best method: pca_kmeans

Ranking:
  1. pca_kmeans: ARI=N/A, sil=0.066, time=0.003s
  2. pathway_gmm: ARI=N/A, sil=0.057, time=0.019s
  3. gene_kmeans: ARI=N/A, sil=0.001, time=0.005s
  4. nmf_clustering: ARI=N/A, sil=-0.016, time=0.026s
  5. random_baseline: ARI=N/A, sil=-0.059, time=0.000s


## 13. Summary & Export

In [20]:
results_summary = {
    'notebook': 'NB18 -- GSE15402 Clinical Phenotype Validation',
    'codeberg_issue': '#16',
    'framework_version': pathway_subtyping.__version__,
    'dataset': {
        'geo_accession': 'GSE15402',
        'citation': 'Hu VW et al. 2009, Autism Research 2(2):78-97 (PMID: 19656385)',
        'platform': 'GPL3427 (TIGR 40K Human array)',
        'tissue': 'lymphoblastoid cell lines (LCL)',
        'design': 'two-channel (Cy3/Cy5), reference design',
        'n_total': int(len(metadata)),
        'n_asd': int(asd_mask.sum()),
        'n_control': int((~asd_mask).sum()),
        'n_genes': int(gene_expression.shape[1]),
        'adi_r_subgroups': {
            'L_language': int((metadata['adi_r_subgroup'] == 'L').sum()),
            'M_mild': int((metadata['adi_r_subgroup'] == 'M').sum()),
            'S_savant': int((metadata['adi_r_subgroup'] == 'S').sum()),
        },
    },
    'probe_mapping': {
        'method': 'MyGene.info (GenBank accession -> HGNC symbol)',
        'total_probes': int(expression_df.shape[0]),
        'probes_with_accession': int(len(probe_acc)),
        'mapped_accessions': int(len(mapping_df)),
        'unique_genes': int(gene_expression.shape[1]),
    },
    'pathway_scoring': {
        'method': 'ssGSEA',
        'database': 'MSigDB Hallmark',
        'n_pathways_scored': int(scoring_result.n_pathways_scored),
        'n_pathways_skipped': int(scoring_result.n_pathways_skipped),
        'gene_coverage': f'{coverage:.1%}',
    },
    'clustering': {
        'algorithm': 'GMM',
        'optimal_k': int(optimal_k),
        'k_range': K_RANGE,
        'min_cluster_fraction': 0.05,
        'silhouette': float(clustering.silhouette),
        'calinski_harabasz': float(clustering.calinski_harabasz),
        'davies_bouldin': float(clustering.davies_bouldin),
        'bic': float(clustering.bic) if clustering.bic is not None else None,
        'converged': bool(clustering.converged),
        'subtype_sizes': {str(i): int((labels == i).sum()) for i in range(optimal_k)},
        'rejected_k': {str(k): reason for k, reason in selection.rejected_k.items()},
    },
    'validation': {
        'gates_passed': int(sum(g.passed for g in val_result.results)),
        'gates_total': int(len(val_result.results)),
        'all_passed': bool(val_result.all_passed),
        'gate_details': [{
            'name': str(g.name),
            'passed': bool(g.passed),
            'metric_name': str(g.metric_name),
            'metric_value': float(g.metric_value),
            'threshold': float(g.threshold),
        } for g in val_result.results],
    },
    'clinical_correlation': {
        'chi2_statistic': float(chi2),
        'chi2_pvalue': float(p_chi2),
        'chi2_df': int(dof),
        'adjusted_rand_index': float(ari),
        'n_fisher_tests': int(n_tests),
        'n_significant_fisher': int(n_sig),
        'fisher_results': fisher_df.to_dict('records'),
        'contingency_table': ct_no_margins.to_dict(),
    },
    'benchmark': {
        'best_method': str(bench_result.best_method),
        'ranking': list(bench_result.ranking),
    },
    'seed': SEED,
}

# Save JSON
json_path = os.path.join(OUTPUT_DIR, 'results_summary.json')
with open(json_path, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print(f'Saved: {json_path}')

# Save CSVs
asd_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_asd.csv'))
pathway_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_all.csv'))
asd_meta.to_csv(os.path.join(OUTPUT_DIR, 'sample_metadata_with_subtypes.csv'))
gene_expression.to_csv(os.path.join(OUTPUT_DIR, 'gene_expression_processed.csv'))
fisher_df.to_csv(os.path.join(OUTPUT_DIR, 'fisher_exact_results.csv'), index=False)

print(f'\nAll outputs saved to {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.0f} KB)')

print(f'\n{"=" * 60}')
print(f'NB18 COMPLETE')
print(f'{"=" * 60}')
print(f'Dataset: GSE15402 ({len(asd_meta)} ASD, {(~asd_mask).sum()} control)')
print(f'Subtypes: k={optimal_k} (GMM, silhouette={clustering.silhouette:.4f})')
print(f'Validation: {sum(g.passed for g in val_result.results)}/{len(val_result.results)} gates passed')
print(f'Clinical: chi2 p={p_chi2:.4f}, ARI={ari:.4f}, {n_sig} significant Fisher pairs')
print(f'Benchmark: best method = {bench_result.best_method}')

Saved: ./outputs/gse15402/results_summary.json

All outputs saved to ./outputs/gse15402/
  benchmark_comparison.png (42 KB)
  cluster_selection.png (91 KB)
  fisher_exact_results.csv (1 KB)
  gene_contributions.csv (9 KB)
  gene_expression_processed.csv (603 KB)
  gene_heatmap.png (162 KB)
  pathway_enrichment.csv (19 KB)
  pathway_scores_all.csv (82 KB)
  pathway_scores_asd.csv (62 KB)
  pathway_scores_matrix.csv (3 KB)
  pca_scatter.png (141 KB)
  results_summary.json (8 KB)
  sample_metadata_with_subtypes.csv (14 KB)
  subtype_heatmap.png (360 KB)
  subtype_summary.csv (0 KB)
  subtypes_vs_adir_heatmap.png (202 KB)

NB18 COMPLETE
Dataset: GSE15402 (87 ASD, 29 control)
Subtypes: k=6 (GMM, silhouette=0.0571)
Validation: 2/3 gates passed
Clinical: chi2 p=0.0010, ARI=0.0568, 0 significant Fisher pairs
Benchmark: best method = pca_kmeans


## References & Data Availability

1. Hu VW, Frank BC, Heine S, Lee NH, Bhagavan S (2006). "Gene expression profiling of lymphoblastoid cell lines from monozygotic twins discordant in severity of autism reveals differential regulation of neurologically relevant genes." *BMC Genomics* 7:118. PMID: [16709250](https://pubmed.ncbi.nlm.nih.gov/16709250/)

2. Hu VW, Nguyen A, Kim KS, Steinberg ME, Sarachana T, Scully MA, Solber SJ, Luu T, Lee NH (2009). "Gene expression profiling of lymphoblasts from autistic and nonaffected sib pairs: altered pathways in neuronal development and steroid biosynthesis." *PLoS ONE* 4(6):e5775. PMID: [19492049](https://pubmed.ncbi.nlm.nih.gov/19492049/)

3. Hu VW, Steinberg ME (2009). "Novel clustering of items from the Autism Diagnostic Interview-Revised to define phenotypes within autism spectrum disorders." *Autism Research* 2(2):67-77. PMID: [19455643](https://pubmed.ncbi.nlm.nih.gov/19455643/)

4. Hu VW, Sarachana T, Kim KS, Nguyen A, Kulber S, Steinberg ME, Luu T, Lai Y, Lee NH (2009). "Gene expression profiling differentiates autism case–controls and phenotypic variants of autism spectrum disorders: evidence for circadian rhythm dysfunction in severe autism." *Autism Research* 2(2):78-97. PMID: [19656385](https://pubmed.ncbi.nlm.nih.gov/19656385/)

5. Liberzon A, Birger C, Thorvaldsdottir H, Ghandi M, Mesirov JP, Tamayo P (2015). "The Molecular Signatures Database (MSigDB) hallmark gene set collection." *Cell Systems* 1(6):417-425.

---

**Data availability:** GSE15402 is freely available from [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE15402). No authentication required.

**Code:** [pathway-subtyping-framework](https://codeberg.org/pathways/pathway-subtyping-framework) (Codeberg)